# 0. Data Preparation

**Dataset:** Genome-scale Perturb-seq — Primary Human CD4+ T Cells (Zhu, Dann et al. 2025)  
**Reference:** https://virtualcellmodels.cziscience.com/dataset/genome-scale-tcell-perturb-seq

This notebook handles all one-time data preparation tasks that do not depend on which condition is being analysed downstream. It only needs to be run once per dataset.

**Part 1, NTC extraction:** Each raw per-donor h5ad file (>100 GB) contains all perturbed and non-targeting control (NTC) cells. We filter to NTC cells only and save a much smaller file per donor per condition. These smaller files are what Notebook 1 uses to compute the stimulation vector.

**Part 2, BTLA reference vector (v_btla):** The BTLA inhibitory vector is derived from an external bulk RNA-seq experiment comparing BTLA signalling vs. TCR stimulation alone (Zhu group, unpublished bulk data). It is condition-agnostic (it does not depend on which Perturb-seq condition we are analysing) so it is computed once here and saved. Notebook 1 reads it directly rather than recomputing it per condition.

**Design note:** v_stim (the stimulation vector) is condition-specific because it measures NTC-Stim vs NTC-Rest, which varies by timepoint. It is computed in Notebook 1. v_btla is fixed regardless of condition and lives here.

## Contents
1. Configuration
2. Part 1: NTC extraction
3. Part 2: BTLA reference vector


### 1. Configuration

In [1]:
import os
print(f'Working directory: {os.getcwd()}')

Working directory: /mnt/R0/Projects/POIAZ/Ilaria/Scripts


In [2]:
import numpy as np
import pandas as pd
import anndata

# Data paths
DATA_DIR = '../Data'
DESTATS_PATH = '../Data/GWCD4i.DE_stats.h5ad'
BTLA_DEG_PATH = '../Data/DEG_wide_statistics_10032026.xlsx'

# Output folder for shared reference files (condition-agnostic outputs)
REF_DIR = '../Results/ref'
os.makedirs(REF_DIR, exist_ok=True)

# NTC extraction settings
# Run one donor+condition at a time — files are >100 GB and cannot be looped
DONOR = 'D1'  # 'D1', 'D2', 'D3', 'D4'
CONDITION = 'Rest'  # 'Rest', 'Stim8hr', 'Stim48hr'

# BTLA vector settings
# The BTLA bulk DEG file contains comparisons at 1h, 4h, and 24h.
# We use 4h as the default: it captures early inhibitory reprogramming
# and is less sparse than 24h (fewer NaN padj values).
# Change this if you want to use a different timepoint.
BTLA_TIMEPOINT = '1h'  # '1h', '4h', '24h'
P_THRESHOLD = 0.05

print(f'Data dir: {DATA_DIR}')
print(f'Ref dir: {REF_DIR}')
print(f'NTC target: {DONOR}_{CONDITION}')
print(f'BTLA tp: {BTLA_TIMEPOINT}')

Data dir: ../Data
Ref dir: ../Results/ref
NTC target: D1_Rest
BTLA tp: 1h


### 2. Part 1: NTC Extraction

Filter the raw assigned-guide h5ad to cells with `guide_type == 'non-targeting'` and save.

**Important:** Run one donor+condition at a time by changing `DONOR` and `CONDITION` in section 1 and re-running this cell. The raw files are >100 GB and loading multiple simultaneously will exhaust RAM.

**Files already extracted (update this list as you go):**
- D1_Rest.NTC.h5ad
- D1_Stim8hr.NTC.h5ad
- D1_Stim48hr.NTC.h5ad
- D2_Rest.NTC.h5ad
- D2_Stim8hr.NTC.h5ad
- D2_Stim48hr.NTC.h5ad
- D3_Rest.NTC.h5ad
- D3_Stim8hr.NTC.h5ad
- D3_Stim48hr.NTC.h5ad
- D4_Rest.NTC.h5ad
- D4_Stim8hr.NTC.h5ad
- D4_Stim48hr.NTC.h5ad

In [3]:
input_file = os.path.join(DATA_DIR, f'{DONOR}_{CONDITION}.assigned_guide.h5ad')
output_file = os.path.join(DATA_DIR, f'{DONOR}_{CONDITION}.NTC.h5ad')

if os.path.exists(output_file):
    print(f'Output already exists: {output_file}')
    print('Delete it manually if you want to re-extract.')
else:
    print(f'Loading: {input_file}')
    adata = anndata.read_h5ad(input_file)
    print(f'Loaded: {adata.shape}')

    if 'guide_type' not in adata.obs.columns:
        raise KeyError('guide_type column not found in adata.obs')

    ntc = adata[adata.obs['guide_type'] == 'non-targeting'].copy()
    print(f'NTC cells: {ntc.n_obs:,} out of {adata.n_obs:,} total')

    ntc.write_h5ad(output_file)
    print(f'Saved: {output_file}')

    del adata, ntc

Output already exists: ../Data/D1_Rest.NTC.h5ad
Delete it manually if you want to re-extract.


### 3. Part 2: BTLA Reference Vector

Build v_btla from the bulk RNA-seq BTLA vs TCR DEG table and save it aligned to the DE_stats gene universe. This only needs to be done once because the BTLA data is external and does not change between Perturb-seq conditions.

**What the BTLA vector represents:** Each entry is the log2 fold-change of a gene in cells with active BTLA signalling relative to TCR stimulation alone, at the chosen timepoint. Genes below the significance threshold are set to zero. The resulting vector defines the inhibitory/exhaustion axis used for DPD_btla scoring in Notebook 1.

**Gene identifier mapping:** The BTLA DEG file uses gene symbols (e.g. CD3E). DE_stats uses Ensembl IDs (e.g. ENSG00000198851). We build the symbol-to-Ensembl map from the NTC h5ad var metadata, which carries both.

In [4]:
btla_df = pd.read_excel(BTLA_DEG_PATH)
print(f'BTLA DEG file: {btla_df.shape}')
print(f'Columns: {btla_df.columns.tolist()}')

fc_col = f'BTLAvsTCR_{BTLA_TIMEPOINT}_log2FoldChange'
padj_col = f'BTLAvsTCR_{BTLA_TIMEPOINT}_padj'
print(f'\nUsing: {fc_col}')
print(f'Non-null padj: {btla_df[padj_col].notna().sum():,}')

btla_sig = btla_df[btla_df[padj_col] <= P_THRESHOLD].copy()
print(f'Significant genes (padj <= {P_THRESHOLD}): {len(btla_sig):,}')

BTLA DEG file: (16725, 13)
Columns: ['gene', 'BTLAvsTCR_1h_log2FoldChange', 'BTLAvsTCR_1h_padj', 'BTLAvsTCR_4h_log2FoldChange', 'BTLAvsTCR_4h_padj', 'BTLAvsTCR_24h_log2FoldChange', 'BTLAvsTCR_24h_padj', 'TCRvsUC_1h_log2FoldChange', 'TCRvsUC_1h_padj', 'TCRvsUC_4h_log2FoldChange', 'TCRvsUC_4h_padj', 'TCRvsUC_24h_log2FoldChange', 'TCRvsUC_24h_padj']

Using: BTLAvsTCR_1h_log2FoldChange
Non-null padj: 16,660
Significant genes (padj <= 0.05): 1


In [5]:
# Load DE_stats var names (Ensembl IDs) and build the symbol -> Ensembl map
# We only need .var so open in backed mode to avoid loading the full matrix
de_stats = anndata.read_h5ad(DESTATS_PATH, backed='r')
gene_ids_de_stats = de_stats.var_names.values
print(f'DE_stats genes: {len(gene_ids_de_stats):,}')
de_stats.file.close()

# Load one NTC file just to get the var table (symbol <-> Ensembl mapping)
ntc_ref_path = os.path.join(DATA_DIR, 'D1_Rest.NTC.h5ad')
_tmp = anndata.read_h5ad(ntc_ref_path, backed='r')
ntc_var = _tmp.var.copy()
_tmp.file.close()
del _tmp
print(f'NTC var columns: {ntc_var.columns.tolist()}')

# Detect symbol column
if 'gene_name' in ntc_var.columns:
    sym_col = 'gene_name'
elif 'gene_symbols' in ntc_var.columns:
    sym_col = 'gene_symbols'
elif 'feature_name' in ntc_var.columns:
    sym_col = 'feature_name'
else:
    sym_col = None
    print('WARNING: no symbol column found, using var index as symbols')

if sym_col:
    symbol_to_ensembl = dict(zip(ntc_var[sym_col].str.upper(), ntc_var.index))
else:
    symbol_to_ensembl = {s.upper(): s for s in ntc_var.index}

print(f'Symbol-to-Ensembl map size: {len(symbol_to_ensembl):,}')

DE_stats genes: 10,282
NTC var columns: ['gene_ids', 'feature_types', 'genome', 'gene_name', 'mt']
Symbol-to-Ensembl map size: 18,130


In [6]:
# Map BTLA gene symbols to Ensembl IDs
btla_sig['ensembl_id'] = btla_sig['gene'].str.upper().map(symbol_to_ensembl)
btla_mapped = btla_sig.dropna(subset=['ensembl_id']).copy()
btla_unmapped = btla_sig[btla_sig['ensembl_id'].isna()]

print(f'Significant BTLA genes: {len(btla_sig):,}')
print(f'Mapped to Ensembl: {len(btla_mapped):,}')
print(f'Unmapped (not in DE_stats): {len(btla_unmapped):,}')
if len(btla_unmapped) > 0:
    print(f'Sample unmapped: {btla_unmapped["gene"].head(10).tolist()}')

# Build v_btla in DE_stats gene order
v_btla = np.zeros(len(gene_ids_de_stats), dtype=np.float32)
n_placed = 0
for _, row in btla_mapped.iterrows():
    ens = row['ensembl_id']
    fc = row[fc_col]
    if pd.isna(fc):
        continue
    idx = np.where(gene_ids_de_stats == ens)[0]
    if len(idx) == 1:
        v_btla[idx[0]] = float(fc)
        n_placed += 1

btla_norm = float(np.linalg.norm(v_btla))
print(f'\nGenes placed in v_btla: {n_placed:,}')
print(f'Non-zero entries: {(v_btla != 0).sum():,}')
print(f'v_btla range: [{v_btla.min():.3f}, {v_btla.max():.3f}]')
print(f'v_btla L2 norm: {btla_norm:.4f}')

if btla_norm == 0:
    raise ValueError('v_btla is all zeros, check BTLA DEG file and gene ID mapping')

Significant BTLA genes: 1
Mapped to Ensembl: 0
Unmapped (not in DE_stats): 1
Sample unmapped: ['RGPD2']

Genes placed in v_btla: 0
Non-zero entries: 0
v_btla range: [0.000, 0.000]
v_btla L2 norm: 0.0000


ValueError: v_btla is all zeros, check BTLA DEG file and gene ID mapping

In [ ]:
# Save v_btla aligned to DE_stats gene order
# gene_name column is the Ensembl ID (matches DE_stats var_names order)
btla_out = pd.DataFrame({
    'ensembl_id': gene_ids_de_stats,
    'logFC_btla': v_btla,
    'nonzero': (v_btla != 0)
})
btla_path = os.path.join(REF_DIR, f'v_btla_{BTLA_TIMEPOINT}.csv')
btla_out.to_csv(btla_path, index=False)
print(f'Saved: {btla_path}')

# Save a metadata file so downstream notebooks know what went into this vector
meta = {
    'btla_timepoint': BTLA_TIMEPOINT,
    'p_threshold': P_THRESHOLD,
    'n_sig_genes': len(btla_sig),
    'n_mapped': len(btla_mapped),
    'n_unmapped': len(btla_unmapped),
    'n_placed_in_vector': n_placed,
    'l2_norm': btla_norm,
    'source_file': BTLA_DEG_PATH,
    'de_stats_gene_universe': DESTATS_PATH,
}
pd.DataFrame([meta]).to_csv(os.path.join(REF_DIR, f'v_btla_{BTLA_TIMEPOINT}_meta.csv'), index=False)
print(f'Saved: {os.path.join(REF_DIR, f"v_btla_{BTLA_TIMEPOINT}_meta.csv")}')
print('\nPart 2 complete.')